Building population-weighted weather index using 

- one representative coordinate per state + DC (lower 48 only + DC only)
- 2020 census as baseline for population weights
- daily mean temperature and hdd/cdd data (based on max,min,65 index)
- calculate state-level before weighting

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from gas_forecast.data.weather import calculate_hdd_cdd
from gas_forecast.data.weather import fetch_all_state_temperatures
from gas_forecast.data.export import save_versioned_parquet

In [ ]:
# load storage data

STORAGE_PATH = Path("../data/processed/lower48_weekly_storage_latest.parquet")
storage = pd.read_parquet(STORAGE_PATH)

# initialize data sources and dates
CENSUS_URL = (
    "https://www2.census.gov/geo/docs/reference/"
    "cenpop2020/CenPop2020_Mean_ST.txt"
)

START_DATE = storage["date"].min().strftime("%Y-%m-%d")
END_DATE = storage["date"].max().strftime("%Y-%m-%d")


In [ ]:
RAW_DIR = Path("../data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

#load census data
state_locations = pd.read_csv(
    CENSUS_URL,
    dtype={"STATEFP": str},
)

EXCLUDED_AREAS = {
    "Alaska",
    "Hawaii",
    "Puerto Rico",
}

state_locations = (
    state_locations
    .loc[~state_locations["STNAME"].isin(EXCLUDED_AREAS)]
    .copy()
)


state_locations["WEIGHT"] = state_locations["POPULATION"] / state_locations["POPULATION"].sum()

state_locations.head()

# validation - should be 48 states + DC and weights sum to 1
# print(state_locations["STNAME"].unique())

# note that the latitude longitude are the population weighted center of the state. this is not ideal for weather modeling (as climates can vary heavily among states) but do solve our issues for now

,STATEFP,STNAME,POPULATION,LATITUDE,LONGITUDE,WEIGHT
0,01,Alabama,5024279,33.016191,-86.753353,0.015259
2,04,Arizona,7151502,33.371388,-111.882468,0.021720
3,05,Arkansas,3011524,35.199251,-92.713212,0.009146
4,06,California,39538223,35.491035,-119.347852,0.120082
5,08,Colorado,5773714,39.534747,-105.185361,0.017535


In [ ]:
WEATHER_CACHE_DIR = RAW_DIR / "weather_cache"

state_daily_weather_raw = fetch_all_state_temperatures(
    locations=state_locations,
    start_date=START_DATE,
    end_date=END_DATE,
    location_batch_size=1,
    date_freq="YS",
    pause_seconds=3.0,
    cache_dir=WEATHER_CACHE_DIR,
)

[1/637] 2014-07-11 to 2014-12-31 | Alabama
[2/637] 2014-07-11 to 2014-12-31 | Arizona


KeyboardInterrupt: 

In [ ]:
state_daily_weather_raw.head()

NameError: name 'state_daily_weather' is not defined

In [ ]:
state_daily_weather_raw.describe()

,date,temperature_f,population,population_weight
count,214081,214081.000000,2.140810e+05,214081.000000
mean,2020-06-26 00:00:00,55.194032,6.719604e+06,0.020408
min,2014-07-04 00:00:00,-33.300000,5.768510e+05,0.001752
25%,2017-06-30 00:00:00,40.600000,1.961504e+06,0.005957
50%,2020-06-26 00:00:00,57.300000,4.657757e+06,0.014146
75%,2023-06-23 00:00:00,71.200000,7.705281e+06,0.023402
max,2026-06-19 00:00:00,104.800000,3.953822e+07,0.120082
std,NaN,19.476062,7.399512e+06,0.022473


In [ ]:
state_daily_weather_raw.dtypes()

NameError: name 'state_daily_weather' is not defined

In [ ]:
state_daily_weather_raw["state"].nunique(), len(state_daily_weather_raw)

(49, 214081)

In [ ]:
save_versioned_parquet(state_daily_weather_raw, output_dir="../data/processed")



In [ ]:
WEATHER_PATH = Path("../data/processed/lower48_daily_weather.parquet")

weather = pd.read_parquet(WEATHER_PATH)
degreedays = calculate_hdd_cdd(weather)


In [6]:
degreedays.head()

,date,temperature_f,state,population,population_weight,hdd,cdd
0,2014-07-04,74.9,Alabama,5024279,0.015259,0.0,9.9
1,2014-07-05,75.8,Alabama,5024279,0.015259,0.0,10.8
2,2014-07-06,78.0,Alabama,5024279,0.015259,0.0,13.0
3,2014-07-07,80.3,Alabama,5024279,0.015259,0.0,15.3
4,2014-07-08,81.3,Alabama,5024279,0.015259,0.0,16.3
